# Data Loaders and a Simple RAG Example with Groq

This notebook teaches how to load **TXT, CSV, HTML, PDF, and Wikipedia** content into LangChain and use that content with a Groq chat model.

## Learning goals

- Understand what a document loader does.
- Apply production practices: typed configuration, validation, logging, retries, and timeouts.
- Understand a LangChain `Document`.
- Know when to use `load()` and `lazy_load()`.
- Pass document content to a Groq model with an LCEL chain.
- Understand the difference between this small grounded example and a complete RAG system.

> Run the cells from top to bottom. Select the **Python 3.13.7 (`.venv`)** kernel first.

## 1. Configure Groq

### What?

`python-dotenv` reads settings from the `.env` file. `ChatGroq` is LangChain's wrapper for Groq chat models.

### When and why?

Use environment variables for API keys and model settings so secrets and configuration are not hardcoded in the notebook.

### How?

Your `.env` file should contain:

```text
GROQ_API_KEY=your_groq_api_key
GROQ_MODEL=qwen/qwen3.8-27b
GROQ_MAX_TOKENS=256
RAG_MAX_CONTEXT_CHARS=8000
HTTP_TIMEOUT_SECONDS=20
```

Do not upload or commit your real `.env` file.

### Production practices used here

| Practice | Why it matters |
| --- | --- |
| Frozen typed `Settings` | Keeps validated configuration in one predictable object |
| Reusable functions | Makes loading logic testable and reusable |
| Explicit validation | Fails early with a useful error message |
| Structured logging | Records operations without printing secrets |
| HTTP timeout and retries | Prevents indefinite waits and handles temporary server failures |
| Narrow exception behavior | Errors remain visible instead of being silently ignored |

In [5]:
import logging
import os
from dataclasses import dataclass
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(), override=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
)
logger = logging.getLogger("rag_demo")


def positive_int_from_env(name: str, default: int) -> int:
    raw_value = os.getenv(name, str(default))
    try:
        value = int(raw_value)
    except ValueError as error:
        raise ValueError(f"{name} must be an integer, received {raw_value!r}.") from error
    if value <= 0:
        raise ValueError(f"{name} must be greater than zero.")
    return value


@dataclass(frozen=True, slots=True)
class Settings:
    groq_model: str
    max_output_tokens: int
    max_context_chars: int
    request_timeout_seconds: int

    @classmethod
    def from_environment(cls) -> "Settings":
        if not os.getenv("GROQ_API_KEY"):
            raise RuntimeError("GROQ_API_KEY is missing from the .env file.")
        return cls(
            groq_model=os.getenv("GROQ_MODEL", "qwen/qwen3.8-27b"),
            max_output_tokens=positive_int_from_env("GROQ_MAX_TOKENS", 256),
            max_context_chars=positive_int_from_env("RAG_MAX_CONTEXT_CHARS", 8000),
            request_timeout_seconds=positive_int_from_env("HTTP_TIMEOUT_SECONDS", 20),
        )


settings = Settings.from_environment()
logger.info("Configuration loaded for model %s", settings.groq_model)

2026-09-30 10:25:13,169 | INFO | rag_demo | Configuration loaded for model qwen/qwen3.8-27b


In [6]:
from langchain_groq import ChatGroq

chat_model = ChatGroq(
    model=settings.groq_model,
    temperature=0.2,
    reasoning_effort="none",
    max_tokens=settings.max_output_tokens,
)

### Important model parameters

| Parameter | What it controls | When to change it |
| --- | --- | --- |
| `model` | Which Groq-hosted model answers | Change when you need different speed, quality, or model features |
| `temperature` | Output randomness | Use a low value for factual RAG answers; increase it for creative writing |
| `reasoning_effort` | Amount of reasoning for models that support it | Use `none` for simple work and a higher supported value for harder work |
| `max_tokens` | Maximum number of tokens generated in the answer | Increase it when answers are being cut off |

> Model support differs. If a selected Groq model rejects `reasoning_effort`, remove that argument or use a compatible model.

In [7]:
messages = [
    ("system", "You are a helpful teacher who explains concepts simply."),
    ("human", "Explain a document loader in one sentence."),
]

response = chat_model.invoke(messages)
print(response.content)

2026-09-30 10:25:25,481 | INFO | httpx | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


A document loader is a tool that reads files from various formats (like PDFs or Word docs) and converts them into a standardized format that AI models can easily process.


## 2. What is a document loader?

A **document loader** reads data from a source and converts it into LangChain `Document` objects.

Each `Document` normally has:

| Part | Meaning | Example |
| --- | --- | --- |
| `page_content` | The text that an LLM can read | Text from a page or CSV row |
| `metadata` | Information describing the text | File name, page number, or URL |

### Why use loaders?

Different sources have different formats. Loaders give your application a common `Document` structure before splitting, embedding, retrieval, or prompting.

## 3. Load a text file

- **When:** The source is a plain `.txt` file.
- **Why:** Python can read plain text directly without a deprecated integration package.
- **How:** Read the file with `Path.read_text()` and place its text and source path in a LangChain `Document`.

In [ ]:
from pathlib import Path
from langchain_core.documents import Document


def require_file(path: str | Path) -> Path:
    file_path = Path(path)
    if not file_path.is_file():
        raise FileNotFoundError(f"Required file was not found: {file_path.resolve()}")
    return file_path


def load_text_document(path: str | Path) -> list[Document]:
    file_path = require_file(path)
    content = file_path.read_text(encoding="utf-8").strip()
    if not content:
        raise ValueError(f"Text file is empty: {file_path}")
    logger.info("Loaded text file: %s", file_path)
    return [Document(page_content=content, metadata={"source": str(file_path)})]


text_documents = load_text_document("data/be-good.txt")

print(f"Documents loaded: {len(text_documents)}")
print("Metadata:", text_documents[0].metadata)
print("Content preview:\n", text_documents[0].page_content[:500])

2026-09-30 10:25:41,068 | INFO | rag_demo | Loaded text file: data\be-good.txt


Documents loaded: 1
Metadata: {'source': 'data\\be-good.txt'}
Content preview:
 Be good

April 2008(This essay is derived from a talk at the 2008 Startup School.)About a month after we started Y Combinator we came up with the
phrase that became our motto: Make something people want.  We've
learned a lot since then, but if I were choosing now that's still
the one I'd pick.Another thing we tell founders is not to worry too much about the
business model, at least at first.  Not because making money is
unimportant, but because it's so much easier than building something
great.A


## 4. Load a large CSV safely

### Loading everything versus reading a small sample

| Method | Behavior | Best use |
| --- | --- | --- |
| `list(reader)` | Loads every row into memory at once | Small files |
| `islice(reader, 5)` | Reads only the requested rows | Large files or previews |

The tree CSV is large, so this demo reads only its first five rows. A CSV loader usually creates one `Document` per row.

In [ ]:
import csv
from itertools import islice


def load_csv_sample(path: str | Path, *, limit: int = 5) -> list[Document]:
    if limit <= 0:
        raise ValueError("CSV sample limit must be greater than zero.")

    file_path = require_file(path)
    documents: list[Document] = []
    with file_path.open(encoding="utf-8-sig", newline="") as csv_file:
        reader = csv.DictReader(csv_file)
        if not reader.fieldnames:
            raise ValueError(f"CSV file has no header row: {file_path}")

        for row_number, row in enumerate(islice(reader, limit), start=1):
            content = "\n".join(f"{key}: {value or ''}" for key, value in row.items())
            documents.append(
                Document(
                    page_content=content,
                    metadata={"source": str(file_path), "row": row_number},
                )
            )

    logger.info("Loaded %d CSV rows from %s", len(documents), file_path)
    return documents


csv_documents = load_csv_sample("data/Street_Tree_List.csv", limit=5)

print(f"Sample rows loaded: {len(csv_documents)}")
print("First row metadata:", csv_documents[0].metadata)
print("First row content:\n", csv_documents[0].page_content)

## 5. Load an HTML file

- **When:** You have a saved web page or another local HTML file.
- **Why:** Beautiful Soup extracts readable text without the deprecated community loader.
- **How:** Parse the file, call `get_text()`, and store the result in a LangChain `Document`.

> This loader parses a local file. Downloading a live web page is a separate step.

In [ ]:
from bs4 import BeautifulSoup
#bs4 means BeautifulSoup4, a library used for parsing HTML and XML documents.


def load_html_document(path: str | Path) -> list[Document]:
    file_path = require_file(path)
    html = file_path.read_text(encoding="utf-8")
    content = BeautifulSoup(html, "html.parser").get_text("\n", strip=True)
    if not content:
        raise ValueError(f"No readable text was extracted from: {file_path}")
    logger.info("Loaded HTML file: %s", file_path)
    return [Document(page_content=content, metadata={"source": str(file_path)})]


html_documents = load_html_document("data/100-startups.html")

print(f"Documents loaded: {len(html_documents)}")
print("Metadata:", html_documents[0].metadata)
print("Content preview:\n", html_documents[0].page_content[:500])

## 6. Load a PDF

- **When:** Your knowledge is stored in PDF files.
- **Why:** `pypdf` extracts text, and we retain the file and page number as metadata.
- **How:** Read the PDF and create one LangChain `Document` per page.

Loading and splitting are different jobs. The loader extracts content. A **text splitter** later divides long content into smaller chunks for retrieval.

In [ ]:
from pypdf import PdfReader


def load_pdf_documents(path: str | Path) -> list[Document]:
    file_path = require_file(path)
    reader = PdfReader(file_path)
    documents = [
        Document(
            page_content=page.extract_text() or "",
            metadata={"source": str(file_path), "page": page_number},
        )
        for page_number, page in enumerate(reader.pages, start=1)
    ]
    if not documents:
        raise ValueError(f"PDF contains no pages: {file_path}")
    logger.info("Loaded %d PDF pages from %s", len(documents), file_path)
    return documents


pdf_documents = load_pdf_documents("data/5pages.pdf")

print(f"PDF pages loaded: {len(pdf_documents)}")
print("First page metadata:", pdf_documents[0].metadata)
print("First page preview:\n", pdf_documents[0].page_content[:500])

## 7. Load Wikipedia content

- **When:** You need a quick public reference for learning or prototyping.
- **Why:** Wikipedia's REST API returns current article text without a deprecated LangChain loader.
- **How:** Request a page summary, then keep its URL and title as document metadata.

This cell needs internet access. Wikipedia can change and is not always an authoritative source, so verify important facts in production applications.

In [ ]:
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


def build_http_session() -> requests.Session:
    retry_policy = Retry(
        total=3,
        backoff_factor=0.5,
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=frozenset({"GET"}),
    )
    session = requests.Session()
    session.headers.update({"User-Agent": "LangChain-learning-notebook/1.0"})
    session.mount("https://", HTTPAdapter(max_retries=retry_policy))
    return session


def load_wikipedia_summary(
    title: str, *, session: requests.Session, timeout_seconds: int
) -> list[Document]:
    if not title.strip():
        raise ValueError("Wikipedia title cannot be empty.")

    endpoint = "https://en.wikipedia.org/api/rest_v1/page/summary/{title}"
    response = session.get(
        endpoint.format(title=requests.utils.quote(title.replace(" ", "_"), safe="")),
        timeout=timeout_seconds,
    )
    response.raise_for_status()
    payload = response.json()

    content = payload.get("extract", "").strip()
    source = payload.get("content_urls", {}).get("desktop", {}).get("page")
    if not content or not source:
        raise ValueError(f"Wikipedia returned an incomplete response for {title!r}.")

    logger.info("Loaded Wikipedia summary: %s", payload.get("title", title))
    return [
        Document(
            page_content=content,
            metadata={"title": payload.get("title", title), "source": source},
        )
    ]


with build_http_session() as http_session:
    wikipedia_documents = load_wikipedia_summary(
        "Tesla, Inc.",
        session=http_session,
        timeout_seconds=settings.request_timeout_seconds,
    )

# Limit this beginner demo so the request stays within Groq's input limits.
# A complete RAG system would split and retrieve relevant chunks instead.
wikipedia_context = wikipedia_documents[0].page_content[: settings.max_context_chars]

print("Metadata:", wikipedia_documents[0].metadata)
print("Content preview:\n", wikipedia_context[:500])

## 8. Ask Groq a question using the loaded context

### Prompt template

A `ChatPromptTemplate` creates consistent messages and safely inserts changing values such as `{question}` and `{context}`.

### LCEL chain

**LCEL** means **LangChain Expression Language**. The `|` operator passes one step's output to the next:

```text
input dictionary → prompt → Groq model → string parser
```

### Output parser

`StrOutputParser` converts the model's message object into a normal Python string.

### Why instruct the model to use only the context?

It reduces unsupported answers. If the provided text does not contain the answer, the model is instructed to say that it does not know.

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

rag_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Answer using only the supplied context. "
            "If the context does not contain the answer, say you do not know.",
        ),
        ("human", "Question: {question}\n\nContext:\n{context}"),
    ]
)

grounded_chain = rag_prompt | chat_model | StrOutputParser()

In [ ]:
def answer_from_context(question: str, context: str) -> str:
    clean_question = question.strip()
    clean_context = context.strip()
    if not clean_question:
        raise ValueError("Question cannot be empty.")
    if not clean_context:
        raise ValueError("Context cannot be empty.")

    logger.info("Requesting a grounded answer from %s", settings.groq_model)
    return grounded_chain.invoke(
        {"question": clean_question, "context": clean_context}
    ).strip()


answer = answer_from_context(
    question="What is Tesla known for?",
    context=wikipedia_context,
)

print(answer)

## 9. Is this complete RAG?

This is a **small grounded-generation example**, not a complete RAG system. We manually give one document to the model.

**RAG** means **Retrieval-Augmented Generation**. A full RAG application retrieves only the most relevant chunks before generating an answer.

| Step | Purpose | Included here? |
| --- | --- | --- |
| Load | Read source data into `Document` objects | Yes |
| Split | Break long documents into useful chunks | No |
| Embed | Convert chunks into numeric vectors | No |
| Store | Save vectors in a vector database | No |
| Retrieve | Find chunks related to the question | No |
| Generate | Ask the LLM using retrieved context | Yes, with manually selected context |

### When should you build full RAG?

Use full RAG when you have many or long documents and cannot send all their content to the model. Retrieval reduces token usage and helps focus the answer on relevant text.

### Full RAG flow

```text
Documents → chunks → embeddings → vector store
                                      ↓
Question → retriever → relevant chunks → prompt → Groq → answer
```